# Alpine Crest Private-Banking Case Narrative

This case-narrative notebook teaches learners to read the private-banking source packs, separate **Public Facts** from **Interpretation For Detection Patterns**, map evidence to a Detection pattern, and write a short investigation note against synthetic **Alpine Crest Private Bank** data.

Educational use only. This notebook supports fraud-detection curriculum design and is not legal, compliance, audit, investment, regulatory, or professional advice. Learner-facing views do not expose the protected answer keys.

## Setup

Load the private-banking generator, learner-facing views, and the Detection pattern registry. Synthetic data uses the `tiny` scale for fast execution.

In [ ]:
from banking_fraud_lab import (
    build_learner_facing_views,
    generate_private_banking_transaction_fraud_world,
)
from banking_fraud_lab.schema import PATTERN_IDS, PROTECTED_SCENARIO_ANSWER_KEYS

print("Frozen private-banking Detection patterns present:", sorted(p for p in PATTERN_IDS if p.startswith("pb_")))

## Generate Learner-Facing Data

The canonical generator retains protected scenario answer keys for internal validation. The learner-facing tables loaded below exclude that protected table, so the investigation note is built from observable evidence only.

In [ ]:
tables = generate_private_banking_transaction_fraud_world(
    seed=42,
    scenario_prevalence=0.2,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables, "Learner-facing views must exclude protected answer keys"
print("Learner-facing tables:", sorted(learner_tables))
transactions = learner_tables["transactions"]
print("Transactions loaded:", len(transactions))
transactions.head()

## Step 1 — Read The Source Packs

Read the two private-banking source packs that anchor this case narrative:

- `docs/cases/source_packs/private-banking-transaction-monitoring.md` supports the `pb_transaction_fraud` Detection pattern (relationship- and transaction-monitoring control failure).
- `docs/cases/source_packs/private-banking-high-value-movement.md` supports the `pb_high_value_movement` Detection pattern (high-value movement read with Banking relationship context).

These packs separate **Public Facts** (about the public source) from **Interpretation For Detection Patterns** (original learner-facing analysis). The case-narrative exercises below ask you to keep that separation in your own reasoning.

## Step 2 — Separate Facts From Interpretation

Using the source packs above, list two or three **facts** you can cite about the public source family (who published it, what it covers at a high level). Then list two or three **interpretations** — analytic claims about how a Detection pattern shows up in observable signals. Keep facts and interpretation in separate lists so a reviewer can see where evidence ends and analysis begins.

The `pb_high_value_movement` and `pb_transaction_fraud` patterns are the relevant Detection patterns here; both are members of the frozen `PATTERN_IDS` registry.

In [ ]:
private_banking_patterns = sorted(p for p in PATTERN_IDS if p.startswith("pb_"))
assert {'pb_high_value_movement', 'pb_transaction_fraud'} <= set(private_banking_patterns)
private_banking_patterns

## Step 3 — Map Evidence To A Detection Pattern

Examine the learner-facing transaction table for relationship-scale context. The raw `transactions` table exposes `account_id`, `amount_chf`, `booked_at`, `channel`, `direction`, and related columns; the relationship-scale features named in the source packs — such as `amount_to_aum_ratio`, `amount_to_relationship_baseline_ratio`, `is_cross_border`, `counterparty_age_days`, and `is_new_counterparty` — are emitted by the private-banking feature module (`notebooks/04_private_banking_feature_engineering/alpine_crest_feature_engineering.ipynb`). The investigation question here is whether high-value movement looks different when read against the Banking relationship baseline rather than as a raw amount.

In [ ]:
relationship_context = transactions.head(10)[[
    col for col in transactions.columns
    if col in {'transaction_id', 'account_id', 'amount_chf', 'channel', 'direction'}
]]
relationship_context

## Step 4 — Write An Investigation Note

Write a short investigation note (four to six sentences) that:

1. Names the Detection pattern (`pb_high_value_movement` or `pb_transaction_fraud`) your evidence maps to.
2. Cites the candidate data signal from the synthetic Alpine Crest data.
3. States one limitation (for example, legitimate high-net-worth cross-border behaviour).
4. Ends with a follow-up question for stakeholder discussion.

Use the glossary exactly: **Client** is the legal customer; **Banking relationship** is the Swiss-bank-style relationship container. Educational framing only — no compliance instruction, and no claim that the synthetic data reconstructs any public matter.

In [ ]:
investigation_note_template = (
    "Detection pattern: {pattern}.\n\n"
    "Candidate signal: one transaction amount is large in raw terms but reads more clearly against "
    "the Banking relationship baseline (amount_to_relationship_baseline_ratio), and it is cross-border. "
    "This is reviewable evidence that the movement is relationship-scale rather than headline-scale.\n\n"
    "Limitation: legitimate high-net-worth cross-border behaviour can share this shape, so the signal "
    "is decision support, not a finding about the Client.\n\n"
    "Follow-up question: how does relationship-manager responsibility at the time of the transaction "
    "change the review priority?"
)
print(investigation_note_template.format(pattern="pb_high_value_movement"))

## Cleanup

This notebook used only learner-facing views. No protected answer keys were loaded, and no real Client, account, or transaction data was used. Re-run from the top to refresh the synthetic data.

In [ ]:
del tables, learner_tables, transactions
print("Case-narrative notebook complete. Learner-facing views only; protected answer keys excluded.")